# Unified 3DGS Augmentation Pipeline
**Mode A** — Synthesis (Zero123++) | **Mode B** — Restoration (ControlNet) | **Mode C** — ViewCrafter

---

## Run order
1. **Cell 1** — Mount Drive
2. **Cell 2** — Set `SCENE_NAME` and `PIPELINE_MODE` (only cell you need to edit)
3. **Cell 3** — Common deps
4. **Cell 4** — Mode A/B setup *(skip if Mode C)*
5. **Cell 5a** — ViewCrafter clone + condacolab *(Mode C only — runtime restarts after this)*
6. **Cell 5b** — Re-mount Drive + re-run Cell 2 + create conda env *(Mode C only, after restart)*
7. **Cell 5c** — Download ViewCrafter model checkpoints *(Mode C only)*
8. **Cell 6** — CUDA submodules + COLMAP *(always)*
9. **Cell 7** — Launch Gradio UI — use the browser interface to run augmentation
10. **Cell 8** → **Cell 9** → **Cell 10** — SfM → Train → Metrics *(run manually after Gradio finishes)*

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Cell 2: CONFIG — only cell you need to edit
import os

# ============================================================
# USER CONFIG
# ============================================================
SCENE_NAME     = "hotdog"      # folder name inside output_train/
PIPELINE_MODE  = "synthesis"   # "synthesis" | "restoration" | "viewcrafter"
RESTORE_PROMPT = "high quality photo, detailed, sharp focus, 8k"

# ── Derived paths — do not edit ──────────────────────────────
DRIVE_BASE = "/content/drive/MyDrive/pythonprojects_2/final_year_project/3D_project"
GS_BASE    = "/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting"

rawdata_path   = f"{DRIVE_BASE}/output_train/{SCENE_NAME}"
processed_path = f"{DRIVE_BASE}/output_processed/{SCENE_NAME}"
filename       = SCENE_NAME

os.makedirs(processed_path, exist_ok=True)
%cd "{DRIVE_BASE}"

print(f"Scene : {SCENE_NAME}")
print(f"Mode  : {PIPELINE_MODE}")
print(f"Input : {rawdata_path}")

In [ ]:
# Cell 3: Common deps
# Run this on first boot AND again after a condacolab restart (Mode C only).
#
# NOTE — torch version and Mode C:
# The 'import torch' here runs in the BASE Colab Python (used by Modes A and B).
# ViewCrafter (Mode C) runs entirely in its own conda env via:
#   /usr/local/envs/viewcrafter_env/bin/python
# That env has its own separate torch — it never touches the base Python torch.
# So there is NO version conflict between the torch here and ViewCrafter's torch.

requirements = f"{DRIVE_BASE}/requirements.txt"
!pip install -r "{requirements}" -q
!pip install -q gradio transformers>=4.38 huggingface-hub accelerate

import torch
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU            : {torch.cuda.get_device_name(0)}")
print("Common deps ready")

In [ ]:
# Cell 4: Mode A/B setup — HuggingFace login + basicsr patch
# Skipped automatically if PIPELINE_MODE == "viewcrafter"
if PIPELINE_MODE in ["synthesis", "restoration"]:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))

    !sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' \
        /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
    print("HuggingFace login OK")
    print("basicsr compatibility patch applied")
else:
    print(f"Skipping Mode A/B setup (PIPELINE_MODE = '{PIPELINE_MODE}')")

---
## Cells 5a–5c: ViewCrafter setup (Mode C only)
**Skip these entirely if using Mode A or B.**

**Cell 5a** installs condacolab — the Colab runtime **restarts automatically** after it runs.

After the restart, run these cells in order:
1. **Cell 1** — Re-mount Drive
2. **Cell 2** — Re-run CONFIG (set `PIPELINE_MODE = "viewcrafter"` again)
3. **Cell 3** — Re-run common deps ⚠️ *the restart wipes pip installs including gradio — must reinstall*
4. **Cell 5a-post** — Completes ViewCrafter env setup (clone repo + create conda env)
5. **Cell 5c** — Download model checkpoints
6. Continue from **Cell 6**

In [ ]:
# Cell 5a: Clone ViewCrafter + install condacolab
# MODE C ONLY — runtime restarts after this cell
if PIPELINE_MODE == "viewcrafter":
    if not os.path.exists('/content/ViewCrafter'):
        !git clone https://github.com/Drexubery/ViewCrafter /content/ViewCrafter
    print("ViewCrafter repo ready")

    print("\nInstalling condacolab — runtime will restart automatically...")
    !pip install condacolab -q
    import condacolab
    condacolab.install()  # <-- triggers automatic restart
else:
    print(f"Skipping Cell 5a (PIPELINE_MODE = '{PIPELINE_MODE}')")

In [ ]:
# Cell 5a-post: [RUN THIS IMMEDIATELY AFTER THE RESTART — Mode C only]
# After condacolab restarts the runtime, all /content/ files are gone.
# This cell re-clones ViewCrafter and creates the conda environment.
# Before running this: re-run Cell 1 (Mount Drive), Cell 2 (CONFIG), Cell 3 (common deps).

if PIPELINE_MODE == "viewcrafter":
    # Re-clone ViewCrafter since /content/ was wiped by the restart
    if not os.path.exists('/content/ViewCrafter'):
        print("Re-cloning ViewCrafter after restart...")
        !git clone https://github.com/Drexubery/ViewCrafter /content/ViewCrafter
    else:
        print("ViewCrafter repo already present")

    # Create the isolated conda environment for ViewCrafter
    # This env gets its own torch (2.1.0) — completely separate from the base Python torch
    print("\nCreating viewcrafter_env (5-10 min)...")
    !conda create -n viewcrafter_env python=3.10 -y -q
    !conda install -n viewcrafter_env \
        -c pytorch -c nvidia -c pytorch3d \
        pytorch=2.1.0 torchvision pytorch-cuda=12.1 pytorch3d \
        -y -q
    !conda run -n viewcrafter_env pip install -q \
        einops imageio imageio-ffmpeg kornia matplotlib moviepy \
        numpy open-clip-torch opencv-python Pillow pytorch-lightning \
        PyYAML roma scikit-image scikit-learn scipy tensorboard \
        timm tqdm "transformers<4.40.0" trimesh omegaconf
    print("\nviewcrafter_env ready — run Cell 5c next (model downloads)")
else:
    print(f"Skipping (PIPELINE_MODE = '{PIPELINE_MODE}')")

In [ ]:
# Cell 5c: Download ViewCrafter model checkpoints (~25GB total)
if PIPELINE_MODE == "viewcrafter":
    import os
    os.makedirs("/content/ViewCrafter/checkpoints", exist_ok=True)

    dust3r = "/content/ViewCrafter/checkpoints/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth"
    vc_ckpt = "/content/ViewCrafter/checkpoints/model_sparse.ckpt"

    if not os.path.exists(dust3r):
        print("Downloading DUSt3R backbone...")
        !wget -q https://download.europe.naverlabs.com/ComputerVision/DUSt3R/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth \
             -P /content/ViewCrafter/checkpoints/
    else:
        print("DUSt3R already present")

    if not os.path.exists(vc_ckpt):
        print("Downloading ViewCrafter sparse model (~23GB)...")
        !wget -q https://huggingface.co/Drexubery/ViewCrafter_25_sparse/resolve/main/model_sparse.ckpt \
             -P /content/ViewCrafter/checkpoints/
    else:
        print("ViewCrafter checkpoint already present")

    print("All checkpoints ready")
else:
    print(f"Skipping Cell 5c (PIPELINE_MODE = '{PIPELINE_MODE}')")

In [ ]:
# Cell 6: CUDA submodules + COLMAP (always run regardless of mode)
import os, shutil
repo_path = GS_BASE

!pip install -q plyfile
!pip uninstall -y tensorflow tensorflow-probability > /dev/null 2>&1
print("TF removed — GPU memory freed")

print("Compiling CUDA extensions on local SSD...")
!rm -rf /content/submodules_local
!cp -r "{repo_path}/submodules" /content/submodules_local
!rm -rf /content/submodules_local/diff-gaussian-rasterization/build
!rm -rf /content/submodules_local/diff-gaussian-rasterization/*.egg-info
!rm -rf /content/submodules_local/simple-knn/build
!rm -rf /content/submodules_local/simple-knn/*.egg-info
!pip install -q /content/submodules_local/diff-gaussian-rasterization
!pip install -q /content/submodules_local/simple-knn
print("CUDA extensions compiled")

print("Installing COLMAP, ffmpeg, xvfb...")
!sudo apt-get install -y ffmpeg colmap xvfb imagemagick > /dev/null 2>&1
!pip install -q pycolmap
print("All infrastructure ready — launch Gradio UI (Cell 7) next")

In [ ]:
# Cell 7: Gradio UI — unified interface for all three pipeline modes
import gradio as gr
import subprocess, os, shutil, glob
from PIL import Image

# Path constants (match Cell 2)
_DRIVE_BASE = "/content/drive/MyDrive/pythonprojects_2/final_year_project/3D_project"
_GS_BASE    = "/content/drive/MyDrive/pythonprojects_2/final_year_project/gaussian-splatting"
_VC_PYTHON  = "/usr/local/envs/viewcrafter_env/bin/python"
_VC_DIR     = "/content/ViewCrafter"


# ── Shared helpers ────────────────────────────────────────────────────────

def _screen(rawdata_path, processed_path):
    """Run process_file.py quality screener. Returns (ok, log_str)."""
    r = subprocess.run(
        ["python", "process_file.py", "--mode", "synthetic",
         "--input_dir", rawdata_path, "--out_dir", processed_path],
        capture_output=True, text=True, cwd=_DRIVE_BASE
    )
    return r.returncode == 0, r.stdout + ("\nERROR:\n" + r.stderr if r.returncode != 0 else "")


def _stage(scene_name, source_dir):
    """Copy PNGs from source_dir to gaussian-splatting input_dataset."""
    dest = f"{_GS_BASE}/input_dataset/{scene_name}/input"
    os.makedirs(dest, exist_ok=True)
    count = 0
    for f in os.listdir(source_dir):
        if f.lower().endswith('.png'):
            shutil.copy2(os.path.join(source_dir, f), dest)
            count += 1
    return dest, count


def _preview(dest, n=16):
    """Return list of PIL Images from dest folder for gallery preview."""
    pngs = sorted(glob.glob(f"{dest}/*.png"))[:n]
    return [Image.open(p) for p in pngs]


def _save_scene(scene_name):
    """Persist scene name so Cells 8-10 can pick it up."""
    import builtins
    builtins._gs_filename = scene_name


# ── Mode A: Synthesis ─────────────────────────────────────────────────────

def run_synthesis(scene_name):
    logs = []
    def log(msg): logs.append(msg); return "\n".join(logs)

    rawdata   = f"{_DRIVE_BASE}/output_train/{scene_name}"
    processed = f"{_DRIVE_BASE}/output_processed/{scene_name}"
    os.makedirs(processed, exist_ok=True)

    if not os.path.exists(rawdata):
        yield log(f"ERROR: {rawdata} not found on Drive."), []
        return

    yield log("[1/3] Running quality screener..."), []
    ok, out = _screen(rawdata, processed)
    yield log(out), []
    if not ok: return

    yield log("[2/3] Running Zero123++ synthesis (10-30 min)..."), []
    r = subprocess.run(
        ["python", "diffusion_script_v0.py",
         "--input_dir", processed, "--out_dir", processed,
         "--mode", "synthesis"],
        capture_output=True, text=True, cwd=_DRIVE_BASE
    )
    yield log(r.stdout[-3000:] + ("\nERROR:\n" + r.stderr[-500:] if r.returncode != 0 else "")), []
    if r.returncode != 0: return

    dest, count = _stage(scene_name, f"{processed}/final_{scene_name}_run")
    _save_scene(scene_name)
    yield log(f"[3/3] Done. {count} images staged to {dest}.\nRun Cell 8 (convert_ai) next."), _preview(dest)


# ── Mode B: Restoration ───────────────────────────────────────────────────

def run_restoration(scene_name, prompt):
    logs = []
    def log(msg): logs.append(msg); return "\n".join(logs)

    rawdata   = f"{_DRIVE_BASE}/output_train/{scene_name}"
    processed = f"{_DRIVE_BASE}/output_processed/{scene_name}"
    os.makedirs(processed, exist_ok=True)

    if not os.path.exists(rawdata):
        yield log(f"ERROR: {rawdata} not found on Drive."), []
        return

    yield log("[1/3] Running quality screener..."), []
    ok, out = _screen(rawdata, processed)
    yield log(out), []
    if not ok: return

    yield log("[2/3] Running ControlNet restoration (10-20 min)..."), []
    r = subprocess.run(
        ["python", "diffusion_script_v0.py",
         "--input_dir", processed, "--out_dir", processed,
         "--mode", "restoration", "--prompt", prompt],
        capture_output=True, text=True, cwd=_DRIVE_BASE
    )
    yield log(r.stdout[-3000:] + ("\nERROR:\n" + r.stderr[-500:] if r.returncode != 0 else "")), []
    if r.returncode != 0: return

    dest, count = _stage(scene_name, f"{processed}/final_{scene_name}_run")
    _save_scene(scene_name)
    yield log(f"[3/3] Done. {count} images staged.\nRun Cell 8 (convert_ai) next."), _preview(dest)


# ── Mode C: ViewCrafter ───────────────────────────────────────────────────

def run_viewcrafter(scene_name, video_length, ddim_steps):
    logs = []
    def log(msg): logs.append(msg); return "\n".join(logs)

    if not os.path.exists(_VC_PYTHON):
        yield log(f"ERROR: {_VC_PYTHON} not found.\nRun Cells 5a-5c (ViewCrafter setup) first."), [], None
        return

    processed  = f"{_DRIVE_BASE}/output_processed/{scene_name}"
    img_dir    = f"{processed}/processed_{scene_name}"
    if not os.path.exists(img_dir):
        img_dir = f"{_DRIVE_BASE}/output_train/{scene_name}/train"
    output_dir = f"{processed}/final_{scene_name}_run"
    render_mp4 = os.path.join(output_dir, "render.mp4")
    os.makedirs(output_dir, exist_ok=True)

    yield log(f"[1/3] Running ViewCrafter ({int(video_length)} frames, {int(ddim_steps)} steps)...\nThis takes 10-20 min."), [], None

    r = subprocess.run([
        _VC_PYTHON, "inference.py",
        "--image_dir",    img_dir,
        "--out_dir",      output_dir,
        "--mode",         "sparse_view_interp",
        "--bg_trd",       "0.2",
        "--seed",         "123",
        "--ckpt_path",    "./checkpoints/model_sparse.ckpt",
        "--config",       "configs/inference_pvd_1024.yaml",
        "--ddim_steps",   str(int(ddim_steps)),
        "--video_length", str(int(video_length)),
        "--device",       "cuda:0",
        "--height",       "576",
        "--width",        "1024",
        "--model_path",   "./checkpoints/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth"
    ], capture_output=True, text=True, cwd=_VC_DIR)

    out_text = r.stdout[-3000:] + ("\nERROR:\n" + r.stderr[-500:] if r.returncode != 0 else "")
    yield log(out_text), [], render_mp4 if os.path.exists(render_mp4) else None
    if r.returncode != 0: return

    # Extract every 4th frame from render.mp4
    yield log("[2/3] Extracting frames from render.mp4..."), [], render_mp4
    frames_dir = os.path.join(output_dir, "extracted_frames")
    os.makedirs(frames_dir, exist_ok=True)
    subprocess.run([
        "ffmpeg", "-y", "-i", render_mp4,
        "-vf", "select=not(mod(n,4))",
        "-vsync", "vfr",
        os.path.join(frames_dir, "frame_%04d.png")
    ], capture_output=True)

    # Stage: original photos + extracted frames
    dest = f"{_GS_BASE}/input_dataset/{scene_name}/input"
    os.makedirs(dest, exist_ok=True)

    orig_train = f"{_DRIVE_BASE}/output_train/{scene_name}/train"
    if os.path.exists(orig_train):
        for f in os.listdir(orig_train):
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                shutil.copy2(os.path.join(orig_train, f), dest)

    for f in os.listdir(frames_dir):
        if f.endswith('.png'):
            shutil.copy2(os.path.join(frames_dir, f), dest)

    total = len(os.listdir(dest))
    _save_scene(scene_name)
    yield log(f"[3/3] Done. {total} images staged (originals + VC frames).\nRun Cell 8 (convert_ai) next."), _preview(dest), render_mp4


# ── Build UI ──────────────────────────────────────────────────────────────
with gr.Blocks(title="3DGS Unified Pipeline") as demo:
    gr.Markdown(
        "## 3DGS Unified Pipeline\n"
        "**Before using:** Run Cells 1–6. "
        "**After Gradio finishes:** Run Cells 8–10 (SfM → Train → Metrics)."
    )

    with gr.Tabs():

        with gr.Tab("Mode A — Synthesis (Zero123++)"):
            gr.Markdown("_Object-centric NeRF datasets. Generates 6 novel views per anchor image. Black background required._")
            a_scene = gr.Textbox(label="Scene Name", value="hotdog",
                                  placeholder="Must match a folder in output_train/")
            a_run   = gr.Button("Run Synthesis Pipeline", variant="primary")
            a_log   = gr.Textbox(label="Live Log", lines=14, interactive=False)
            a_gal   = gr.Gallery(label="Staged Output Preview", columns=4, height=400)
            a_run.click(fn=run_synthesis, inputs=[a_scene], outputs=[a_log, a_gal])

        with gr.Tab("Mode B — Restoration (ControlNet)"):
            gr.Markdown("_Natural or degraded images with blur/noise. Repairs quality, keeps background._")
            b_scene  = gr.Textbox(label="Scene Name", value="train",
                                   placeholder="Must match a folder in output_train/")
            b_prompt = gr.Textbox(
                label="ControlNet Prompt",
                value="high quality photo, detailed, sharp focus, 8k"
            )
            b_run = gr.Button("Run Restoration Pipeline", variant="primary")
            b_log = gr.Textbox(label="Live Log", lines=14, interactive=False)
            b_gal = gr.Gallery(label="Staged Output Preview", columns=4, height=400)
            b_run.click(fn=run_restoration, inputs=[b_scene, b_prompt], outputs=[b_log, b_gal])

        with gr.Tab("Mode C — ViewCrafter (Natural Scenes)"):
            gr.Markdown(
                "_Natural scene video diffusion. Requires Cells 5a-5c to have run.\n"
                "Outputs: original photos + sampled frames staged together for COLMAP._"
            )
            c_scene = gr.Textbox(label="Scene Name", value="train",
                                  placeholder="Must match a folder in output_train/")
            with gr.Row():
                c_len   = gr.Slider(10, 50, value=25, step=5,
                                     label="Video Length (total output frames)")
                c_steps = gr.Slider(20, 80, value=50, step=10, label="DDIM Steps")
            c_run   = gr.Button("Run ViewCrafter Pipeline", variant="primary")
            c_log   = gr.Textbox(label="Live Log", lines=14, interactive=False)
            c_video = gr.Video(label="render.mp4 Preview")
            c_gal   = gr.Gallery(label="Staged Frames Preview", columns=4, height=400)
            c_run.click(
                fn=run_viewcrafter,
                inputs=[c_scene, c_len, c_steps],
                outputs=[c_log, c_gal, c_video]
            )

demo.queue()
demo.launch(share=True)  # prints public URL valid for 72 hours

---
## Manual Steps — Run after Gradio UI reports 'Done'
Cells 8–10 are always the same regardless of which mode you used.

In [ ]:
# Cell 8: Pose estimation — SuperPoint + LightGlue + COLMAP
import os, builtins
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

# Pick up scene name set by Gradio; fall back to CONFIG value
filename = getattr(builtins, '_gs_filename', SCENE_NAME)
print(f"Processing scene: {filename}")

drive_path = f"{GS_BASE}/input_dataset/{filename}"
local_path = f"/content/local_workspace/{filename}"

print("Transferring dataset to local SSD...")
!mkdir -p "{local_path}"
!rm -rf "{local_path}/input" && mkdir -p "{local_path}/input"
!cp -r "{drive_path}/input/"* "{local_path}/input/"

print("Cleaning workspace...")
!rm -rf "{local_path}/sparse" "{local_path}/distorted"
!rm -f "{local_path}/database.db"

print("Running convert_ai.py (SuperPoint + LightGlue exhaustive matching)...")
%cd {GS_BASE}
!python convert_ai.py --source_path "{local_path}"

print("Syncing sparse/0 back to Drive...")
!cp -r "{local_path}"/* "{drive_path}/"
print(f"Done — {filename} ready for train.py")

In [ ]:
# Cell 9: 3DGS Training — 30,000 iterations
import shutil, os

LOCAL_INPUT  = f"/content/local_workspace/{filename}"
LOCAL_OUTPUT = f"/content/local_workspace/{filename}_final_run"
DRIVE_OUT    = f"{GS_BASE}/output/{filename}_final_run"

print("Staging COLMAP data for training...")
!rm -rf "{LOCAL_INPUT}" && mkdir -p "{LOCAL_INPUT}"
!cp -r "{GS_BASE}/input_dataset/{filename}/input" "{LOCAL_INPUT}/images"
!cp -r "{GS_BASE}/input_dataset/{filename}/sparse" "{LOCAL_INPUT}/"

print("Sanity check — sparse/0 contents:")
!ls -lh "{LOCAL_INPUT}/sparse/0"

print("Starting training (30k iterations, ~55 min)...")
%cd {GS_BASE}
!python train.py \
    -s "{LOCAL_INPUT}" \
    -m "{LOCAL_OUTPUT}" \
    --eval \
    --opacity_reset_interval 9000

print("Saving model to Drive...")
shutil.copytree(LOCAL_OUTPUT, DRIVE_OUT, dirs_exist_ok=True)
print(f"Model saved to {DRIVE_OUT}")

In [ ]:
# Cell 10: Render held-out views + compute PSNR / SSIM / LPIPS
import os

LOCAL_INPUT = f"/content/local_workspace/{filename}"
DRIVE_OUT   = f"{GS_BASE}/output/{filename}_final_run"

print("Re-staging dataset for rendering...")
!rm -rf "{LOCAL_INPUT}" && mkdir -p "{LOCAL_INPUT}"
!cp -r "{GS_BASE}/input_dataset/{filename}/input" "{LOCAL_INPUT}/images"
!cp -r "{GS_BASE}/input_dataset/{filename}/sparse" "{LOCAL_INPUT}/"

%cd {GS_BASE}

print("Rendering test views...")
!python render.py \
    -m "{DRIVE_OUT}" \
    -s "{LOCAL_INPUT}" \
    --skip_train

print("Computing metrics (PSNR / SSIM / LPIPS)...")
!python metrics.py -m "{DRIVE_OUT}"